In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Crime Data Analysis and Prediction")
    .master("local[*]")
    .config("spark.python.worker.reuse", "true")
    .getOrCreate()
)

print("Python:", sys.version)
print("Spark:", spark.version)

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Spark: 4.2.0


In [4]:
data = [
    ("THEFT", "District 1"),
    ("BATTERY", "District 2"),
    ("THEFT", "District 1"),
    ("ROBBERY", "District 7")
]

columns = ["Crime_Type", "District"]

test_df = spark.createDataFrame(data, columns)

test_df.show()

+----------+----------+
|Crime_Type|  District|
+----------+----------+
|     THEFT|District 1|
|   BATTERY|District 2|
|     THEFT|District 1|
|   ROBBERY|District 7|
+----------+----------+



In [5]:
test_df.groupBy("Crime_Type").count().show()

+----------+-----+
|Crime_Type|count|
+----------+-----+
|     THEFT|    2|
|   BATTERY|    1|
|   ROBBERY|    1|
+----------+-----+



# 1. Dataset Loading
## Loading the Chicago Crime Dataset using Apache Spark

In [2]:
crime_df = spark.read.csv(
    "data/Crimes_-_2001_to_Present.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [7]:
crime_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [8]:
total_records = crime_df.count()

print("Total crime records:", total_records)

Total crime records: 7784664


In [9]:
crime_df.show(5, truncate=False)

+--------+-----------+----------------------+---------------------+----+------------+-----------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|ID      |Case Number|Date                  |Block                |IUCR|Primary Type|Description            |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location                     |
+--------+-----------+----------------------+---------------------+----+------------+-----------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|10224738|HY411648   |09/05/2015 01:30:00 PM|043XX S WOOD ST      |0486|BATTERY     |DOMESTIC BAT

# 2. Data Understanding and Quality Analysis
## 2.1 Dataset Dimensions

In [10]:
# Number of rows
num_rows = crime_df.count()

# Number of columns
num_columns = len(crime_df.columns)

print("Number of rows:", num_rows)
print("Number of columns:", num_columns)

Number of rows: 7784664
Number of columns: 22


## 2.2 Dataset Columns


In [11]:
for i, column in enumerate(crime_df.columns, start=1):
    print(f"{i}. {column}")

1. ID
2. Case Number
3. Date
4. Block
5. IUCR
6. Primary Type
7. Description
8. Location Description
9. Arrest
10. Domestic
11. Beat
12. District
13. Ward
14. Community Area
15. FBI Code
16. X Coordinate
17. Y Coordinate
18. Year
19. Updated On
20. Latitude
21. Longitude
22. Location


## 2.3 Dataset Schema

In [12]:
crime_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



## 2.4 Sample Records

In [13]:
crime_df.show(10, truncate=False)

+--------+-----------+----------------------+-----------------------+----+------------------+-----------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|ID      |Case Number|Date                  |Block                  |IUCR|Primary Type      |Description                        |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location                     |
+--------+-----------+----------------------+-----------------------+----+------------------+-----------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|10224738|HY411648   |09/05/2015 01:3

## 2.5 Missing-Value Analysis

In [14]:
from pyspark.sql.functions import col, sum, when, count

missing_values = crime_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in crime_df.columns
])

missing_values.show(truncate=False)

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|ID |Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|Ward  |Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|0  |4          |0   |0    |0   |0           |0          |10381               |0     |0       |0   |47      |614848|613476        |0       |86848       |86848       |0   |0         |86848   |86848    |86848   |
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+---

## 2.6 Missing-Value Percentage

In [15]:
from pyspark.sql.functions import col, sum, when, round

total_records = crime_df.count()

missing_percentage = crime_df.select([
    round(
        sum(when(col(c).isNull(), 1).otherwise(0)) / total_records * 100,
        2
    ).alias(c)
    for c in crime_df.columns
])

missing_percentage.show(truncate=False)

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|ID |Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|0.0|0.0        |0.0 |0.0  |0.0 |0.0         |0.0        |0.13                |0.0   |0.0     |0.0 |0.0     |7.9 |7.88          |0.0     |1.12        |1.12        |0.0 |0.0       |1.12    |1.12     |1.12    |
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+

In [18]:
from pyspark.sql.functions import col, sum, when
import builtins

total_records = crime_df.count()

missing_data = crime_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in crime_df.columns
])

missing_rows = missing_data.collect()[0]

missing_summary = []

for column, count_missing in zip(crime_df.columns, missing_rows):
    if count_missing > 0:
        percentage = (count_missing / total_records) * 100
        
        missing_summary.append(
            (column, int(count_missing), builtins.round(percentage, 2))
        )

missing_df = spark.createDataFrame(
    missing_summary,
    ["Column", "Missing_Count", "Missing_Percentage"]
)

missing_df.orderBy(
    col("Missing_Percentage").desc()
).show(truncate=False)

+--------------------+-------------+------------------+
|Column              |Missing_Count|Missing_Percentage|
+--------------------+-------------+------------------+
|Ward                |614848       |7.9               |
|Community Area      |613476       |7.88              |
|X Coordinate        |86848        |1.12              |
|Location            |86848        |1.12              |
|Latitude            |86848        |1.12              |
|Longitude           |86848        |1.12              |
|Y Coordinate        |86848        |1.12              |
|Location Description|10381        |0.13              |
|District            |47           |0.0               |
|Case Number         |4            |0.0               |
+--------------------+-------------+------------------+



## 2.7 Approximate Record Analysis

In [4]:
from pyspark.sql.functions import approx_count_distinct

total_records = crime_df.count()

approx_unique_ids = crime_df.select(
    approx_count_distinct("ID", rsd=0.01).alias("approx_unique_ids")
).collect()[0]["approx_unique_ids"]

print("Total records:", total_records)
print("Approximate unique IDs:", approx_unique_ids)

Total records: 7784664
Approximate unique IDs: 7818643


## 2.8 Categorical Value Analysis
### Crime Types

In [6]:
from pyspark.sql.functions import col
crime_df.groupBy("Primary Type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

+--------------------------------+-------+
|Primary Type                    |count  |
+--------------------------------+-------+
|THEFT                           |1642148|
|BATTERY                         |1422913|
|CRIMINAL DAMAGE                 |887266 |
|NARCOTICS                       |747633 |
|ASSAULT                         |507296 |
|OTHER OFFENSE                   |483642 |
|BURGLARY                        |424397 |
|MOTOR VEHICLE THEFT             |375495 |
|DECEPTIVE PRACTICE              |344940 |
|ROBBERY                         |292334 |
|CRIMINAL TRESPASS               |214316 |
|WEAPONS VIOLATION               |106418 |
|PROSTITUTION                    |69840  |
|OFFENSE INVOLVING CHILDREN      |55719  |
|PUBLIC PEACE VIOLATION          |52325  |
|SEX OFFENSE                     |30683  |
|CRIM SEXUAL ASSAULT             |27631  |
|INTERFERENCE WITH PUBLIC OFFICER|18392  |
|LIQUOR LAW VIOLATION            |14901  |
|GAMBLING                        |14618  |
+----------

### Arrest Distribution

In [7]:
crime_df.groupBy("Arrest") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+------+-------+
|Arrest|  count|
+------+-------+
| false|5749900|
|  true|2034764|
+------+-------+



In [8]:
crime_df.groupBy("Domestic") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+--------+-------+
|Domestic|  count|
+--------+-------+
|   false|6708370|
|    true|1076294|
+--------+-------+



In [9]:
crime_df.select(
    "Latitude",
    "Longitude",
    "X Coordinate",
    "Y Coordinate"
).summary().show()

+-------+------------------+-------------------+------------------+------------------+
|summary|          Latitude|          Longitude|      X Coordinate|      Y Coordinate|
+-------+------------------+-------------------+------------------+------------------+
|  count|           7697816|            7697816|           7697816|           7697816|
|   mean|41.842183638280936| -87.67149303901566|1164601.2705625854|1885782.8573099175|
| stddev|0.0887959834316862|0.06108257002119852|16846.578922131077|32275.312527212514|
|    min|      36.619446395|      -91.686565684|                 0|                 0|
|    25%|      41.768705597|      -87.713673374|           1152976|           1859069|
|    50%|      41.855906242|      -87.665844597|           1166110|           1890728|
|    75%|      41.906765676|      -87.628194981|           1176371|           1909272|
|    max|      42.022910333|      -87.524529378|           1205119|           1951622|
+-------+------------------+---------------

# 3. Data Cleaning

## 3.1 Handling Missing Categorical Values

In [14]:
from pyspark.sql.functions import col

crime_clean_df = crime_df.fillna({
    "Location Description": "Unknown"
})

print("Categorical missing values handled.")

Categorical missing values handled.


In [15]:
crime_clean_df.select(
    "Location Description",
    "District",
    "Ward",
    "Community Area"
).show(10, truncate=False)

+--------------------+--------+----+--------------+
|Location Description|District|Ward|Community Area|
+--------------------+--------+----+--------------+
|RESIDENCE           |9       |12  |61            |
|CTA BUS             |15      |29  |25            |
|RESIDENCE           |6       |8   |44            |
|SIDEWALK            |14      |35  |21            |
|APARTMENT           |15      |28  |25            |
|RESIDENCE           |6       |21  |71            |
|RESIDENCE-GARAGE    |14      |32  |24            |
|GROCERY FOOD STORE  |10      |25  |31            |
|STREET              |12      |27  |27            |
|Unknown             |8       |15  |63            |
+--------------------+--------+----+--------------+
only showing top 10 rows


In [16]:
from pyspark.sql.functions import sum, when

crime_clean_df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in [
        "Location Description",
        "District",
        "Ward",
        "Community Area"
    ]
]).show()

+--------------------+--------+------+--------------+
|Location Description|District|  Ward|Community Area|
+--------------------+--------+------+--------------+
|                   0|      47|614848|        613476|
+--------------------+--------+------+--------------+



In [17]:
original_count = crime_df.count()
cleaned_count = crime_clean_df.count()

print("Original records:", original_count)
print("Cleaned records:", cleaned_count)
print("Records removed:", original_count - cleaned_count)

Original records: 7784664
Cleaned records: 7784664
Records removed: 0


In [18]:
crime_clean_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = false)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



## 3.2 Converting Date to Timestamp

In [19]:
from pyspark.sql.functions import to_timestamp

crime_clean_df = crime_clean_df.withColumn(
    "Date",
    to_timestamp("Date", "MM/dd/yyyy hh:mm:ss a")
)

crime_clean_df.select("Date").show(10, truncate=False)

+-------------------+
|Date               |
+-------------------+
|2015-09-05 13:30:00|
|2015-09-04 11:30:00|
|2018-09-01 00:01:00|
|2015-09-05 12:45:00|
|2015-09-05 13:00:00|
|2015-09-05 10:55:00|
|2015-09-04 18:00:00|
|2015-09-05 13:00:00|
|2015-09-05 11:30:00|
|2016-05-01 00:25:00|
+-------------------+
only showing top 10 rows


### 3.3 Extracting Temporal Features

In [20]:
from pyspark.sql.functions import year, month, dayofmonth, dayofweek, hour

crime_clean_df = (
    crime_clean_df
    .withColumn("Crime_Year", year("Date"))
    .withColumn("Crime_Month", month("Date"))
    .withColumn("Crime_Day", dayofmonth("Date"))
    .withColumn("Crime_DayOfWeek", dayofweek("Date"))
    .withColumn("Crime_Hour", hour("Date"))
)

crime_clean_df.select(
    "Date",
    "Crime_Year",
    "Crime_Month",
    "Crime_Day",
    "Crime_DayOfWeek",
    "Crime_Hour"
).show(10, truncate=False)

+-------------------+----------+-----------+---------+---------------+----------+
|Date               |Crime_Year|Crime_Month|Crime_Day|Crime_DayOfWeek|Crime_Hour|
+-------------------+----------+-----------+---------+---------------+----------+
|2015-09-05 13:30:00|2015      |9          |5        |7              |13        |
|2015-09-04 11:30:00|2015      |9          |4        |6              |11        |
|2018-09-01 00:01:00|2018      |9          |1        |7              |0         |
|2015-09-05 12:45:00|2015      |9          |5        |7              |12        |
|2015-09-05 13:00:00|2015      |9          |5        |7              |13        |
|2015-09-05 10:55:00|2015      |9          |5        |7              |10        |
|2015-09-04 18:00:00|2015      |9          |4        |6              |18        |
|2015-09-05 13:00:00|2015      |9          |5        |7              |13        |
|2015-09-05 11:30:00|2015      |9          |5        |7              |11        |
|2016-05-01 00:2

In [21]:
crime_clean_df.select(
    "Crime_Year",
    "Crime_Month",
    "Crime_Day",
    "Crime_DayOfWeek",
    "Crime_Hour"
).summary().show()

+-------+-----------------+------------------+------------------+------------------+------------------+
|summary|       Crime_Year|       Crime_Month|         Crime_Day|   Crime_DayOfWeek|        Crime_Hour|
+-------+-----------------+------------------+------------------+------------------+------------------+
|  count|          7784664|           7784664|           7784664|           7784664|           7784664|
|   mean|2009.944267472559| 6.533680323261223|15.612338824129083| 4.039084795438827|13.126249122633938|
| stddev|6.260628233016062|3.3556446007402645|  8.83822737117765|1.9907782768774953| 6.747187146702668|
|    min|             2001|                 1|                 1|                 1|                 0|
|    25%|             2005|                 4|                 8|                 2|                 9|
|    50%|             2009|                 7|                16|                 4|                14|
|    75%|             2015|                 9|                23

In [22]:
print("Records after feature engineering:", crime_clean_df.count())

Records after feature engineering: 7784664


### 3.4 Day of Week Name

In [23]:
from pyspark.sql.functions import date_format

crime_clean_df = crime_clean_df.withColumn(
    "Day_Name",
    date_format("Date", "EEEE")
)

crime_clean_df.select(
    "Date",
    "Crime_DayOfWeek",
    "Day_Name"
).show(10, truncate=False)

+-------------------+---------------+--------+
|Date               |Crime_DayOfWeek|Day_Name|
+-------------------+---------------+--------+
|2015-09-05 13:30:00|7              |Saturday|
|2015-09-04 11:30:00|6              |Friday  |
|2018-09-01 00:01:00|7              |Saturday|
|2015-09-05 12:45:00|7              |Saturday|
|2015-09-05 13:00:00|7              |Saturday|
|2015-09-05 10:55:00|7              |Saturday|
|2015-09-04 18:00:00|6              |Friday  |
|2015-09-05 13:00:00|7              |Saturday|
|2015-09-05 11:30:00|7              |Saturday|
|2016-05-01 00:25:00|1              |Sunday  |
+-------------------+---------------+--------+
only showing top 10 rows


In [24]:
from pyspark.sql.functions import col, sum, when

date_null_count = crime_clean_df.select(
    sum(when(col("Date").isNull(), 1).otherwise(0)).alias("Invalid_Date_Count")
).collect()[0]["Invalid_Date_Count"]

print("Invalid/NULL dates:", date_null_count)

Invalid/NULL dates: 0


In [25]:
crime_clean_df.groupBy("Crime_Year") \
    .count() \
    .orderBy("Crime_Year") \
    .show(100)

+----------+------+
|Crime_Year| count|
+----------+------+
|      2001|485878|
|      2002|486802|
|      2003|475979|
|      2004|469421|
|      2005|453771|
|      2006|448174|
|      2007|437084|
|      2008|427167|
|      2009|392819|
|      2010|370496|
|      2011|351964|
|      2012|336262|
|      2013|307468|
|      2014|275731|
|      2015|264755|
|      2016|269786|
|      2017|269071|
|      2018|268773|
|      2019|261245|
|      2020|212092|
|      2021|208571|
|      2022|238215|
|      2023| 73140|
+----------+------+



In [26]:
crime_clean_df.groupBy("Crime_Month") \
    .count() \
    .orderBy("Crime_Month") \
    .show()

+-----------+------+
|Crime_Month| count|
+-----------+------+
|          1|621870|
|          2|547454|
|          3|649788|
|          4|641066|
|          5|682819|
|          6|681640|
|          7|717118|
|          8|710301|
|          9|668097|
|         10|676128|
|         11|608818|
|         12|579565|
+-----------+------+



In [27]:
crime_clean_df.groupBy("Day_Name") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+---------+-------+
| Day_Name|  count|
+---------+-------+
|   Friday|1169913|
|Wednesday|1119819|
| Saturday|1114957|
|  Tuesday|1112642|
| Thursday|1108515|
|   Monday|1100146|
|   Sunday|1058672|
+---------+-------+



In [28]:
from pyspark.sql.functions import col, sum, when

date_null_count = crime_clean_df.select(
    sum(when(col("Date").isNull(), 1).otherwise(0)).alias("Invalid_Date_Count")
).collect()[0]["Invalid_Date_Count"]

print("Invalid/NULL dates:", date_null_count)

Invalid/NULL dates: 0


In [29]:
print("Total records:", crime_clean_df.count())

Total records: 7784664


# 4. Exploratory Data Analysis

## 4.1 Crime Trend by Year

In [30]:
yearly_crime = (
    crime_clean_df
    .groupBy("Crime_Year")
    .count()
    .orderBy("Crime_Year")
)

yearly_crime.show(100)

+----------+------+
|Crime_Year| count|
+----------+------+
|      2001|485878|
|      2002|486802|
|      2003|475979|
|      2004|469421|
|      2005|453771|
|      2006|448174|
|      2007|437084|
|      2008|427167|
|      2009|392819|
|      2010|370496|
|      2011|351964|
|      2012|336262|
|      2013|307468|
|      2014|275731|
|      2015|264755|
|      2016|269786|
|      2017|269071|
|      2018|268773|
|      2019|261245|
|      2020|212092|
|      2021|208571|
|      2022|238215|
|      2023| 73140|
+----------+------+



In [31]:
yearly_crime.orderBy(
    col("count").desc()
).show(10)

+----------+------+
|Crime_Year| count|
+----------+------+
|      2002|486802|
|      2001|485878|
|      2003|475979|
|      2004|469421|
|      2005|453771|
|      2006|448174|
|      2007|437084|
|      2008|427167|
|      2009|392819|
|      2010|370496|
+----------+------+
only showing top 10 rows


In [32]:
yearly_crime.orderBy(
    col("count").asc()
).show(10)

+----------+------+
|Crime_Year| count|
+----------+------+
|      2023| 73140|
|      2021|208571|
|      2020|212092|
|      2022|238215|
|      2019|261245|
|      2015|264755|
|      2018|268773|
|      2017|269071|
|      2016|269786|
|      2014|275731|
+----------+------+
only showing top 10 rows


In [33]:
yearly_crime.orderBy("Crime_Year").show(100)

+----------+------+
|Crime_Year| count|
+----------+------+
|      2001|485878|
|      2002|486802|
|      2003|475979|
|      2004|469421|
|      2005|453771|
|      2006|448174|
|      2007|437084|
|      2008|427167|
|      2009|392819|
|      2010|370496|
|      2011|351964|
|      2012|336262|
|      2013|307468|
|      2014|275731|
|      2015|264755|
|      2016|269786|
|      2017|269071|
|      2018|268773|
|      2019|261245|
|      2020|212092|
|      2021|208571|
|      2022|238215|
|      2023| 73140|
+----------+------+



In [34]:
crime_clean_df.select(
    "Crime_Year"
).distinct().orderBy("Crime_Year").show(100)

+----------+
|Crime_Year|
+----------+
|      2001|
|      2002|
|      2003|
|      2004|
|      2005|
|      2006|
|      2007|
|      2008|
|      2009|
|      2010|
|      2011|
|      2012|
|      2013|
|      2014|
|      2015|
|      2016|
|      2017|
|      2018|
|      2019|
|      2020|
|      2021|
|      2022|
|      2023|
+----------+



In [35]:
crime_clean_df.select("Date") \
    .orderBy(col("Date").desc()) \
    .show(20, truncate=False)

+-------------------+
|Date               |
+-------------------+
|2023-04-21 23:59:00|
|2023-04-21 23:58:00|
|2023-04-21 23:49:00|
|2023-04-21 23:45:00|
|2023-04-21 23:40:00|
|2023-04-21 23:37:00|
|2023-04-21 23:35:00|
|2023-04-21 23:34:00|
|2023-04-21 23:31:00|
|2023-04-21 23:30:00|
|2023-04-21 23:30:00|
|2023-04-21 23:30:00|
|2023-04-21 23:30:00|
|2023-04-21 23:30:00|
|2023-04-21 23:20:00|
|2023-04-21 23:15:00|
|2023-04-21 23:15:00|
|2023-04-21 23:14:00|
|2023-04-21 23:13:00|
|2023-04-21 23:10:00|
+-------------------+
only showing top 20 rows


In [36]:
crime_type_counts = (
    crime_clean_df
    .groupBy("Primary Type")
    .count()
    .orderBy(col("count").desc())
)

crime_type_counts.show(50, truncate=False)

+---------------------------------+-------+
|Primary Type                     |count  |
+---------------------------------+-------+
|THEFT                            |1642148|
|BATTERY                          |1422913|
|CRIMINAL DAMAGE                  |887266 |
|NARCOTICS                        |747633 |
|ASSAULT                          |507296 |
|OTHER OFFENSE                    |483642 |
|BURGLARY                         |424397 |
|MOTOR VEHICLE THEFT              |375495 |
|DECEPTIVE PRACTICE               |344940 |
|ROBBERY                          |292334 |
|CRIMINAL TRESPASS                |214316 |
|WEAPONS VIOLATION                |106418 |
|PROSTITUTION                     |69840  |
|OFFENSE INVOLVING CHILDREN       |55719  |
|PUBLIC PEACE VIOLATION           |52325  |
|SEX OFFENSE                      |30683  |
|CRIM SEXUAL ASSAULT              |27631  |
|INTERFERENCE WITH PUBLIC OFFICER |18392  |
|LIQUOR LAW VIOLATION             |14901  |
|GAMBLING                       

In [37]:
crime_type_counts.limit(10).show(truncate=False)

+-------------------+-------+
|Primary Type       |count  |
+-------------------+-------+
|THEFT              |1642148|
|BATTERY            |1422913|
|CRIMINAL DAMAGE    |887266 |
|NARCOTICS          |747633 |
|ASSAULT            |507296 |
|OTHER OFFENSE      |483642 |
|BURGLARY           |424397 |
|MOTOR VEHICLE THEFT|375495 |
|DECEPTIVE PRACTICE |344940 |
|ROBBERY            |292334 |
+-------------------+-------+



In [38]:
from pyspark.sql.functions import round

crime_type_percentage = (
    crime_type_counts
    .withColumn(
        "Percentage",
        round(
            (col("count") / total_records) * 100,
            2
        )
    )
)

crime_type_percentage.show(50, truncate=False)

+---------------------------------+-------+----------+
|Primary Type                     |count  |Percentage|
+---------------------------------+-------+----------+
|THEFT                            |1642148|21.09     |
|BATTERY                          |1422913|18.28     |
|CRIMINAL DAMAGE                  |887266 |11.4      |
|NARCOTICS                        |747633 |9.6       |
|ASSAULT                          |507296 |6.52      |
|OTHER OFFENSE                    |483642 |6.21      |
|BURGLARY                         |424397 |5.45      |
|MOTOR VEHICLE THEFT              |375495 |4.82      |
|DECEPTIVE PRACTICE               |344940 |4.43      |
|ROBBERY                          |292334 |3.76      |
|CRIMINAL TRESPASS                |214316 |2.75      |
|WEAPONS VIOLATION                |106418 |1.37      |
|PROSTITUTION                     |69840  |0.9       |
|OFFENSE INVOLVING CHILDREN       |55719  |0.72      |
|PUBLIC PEACE VIOLATION           |52325  |0.67      |
|SEX OFFEN

In [40]:
monthly_crime = (
    crime_clean_df
    .groupBy("Crime_Month")
    .count()
    .orderBy("Crime_Month")
)

monthly_crime.show()

+-----------+------+
|Crime_Month| count|
+-----------+------+
|          1|621870|
|          2|547454|
|          3|649788|
|          4|641066|
|          5|682819|
|          6|681640|
|          7|717118|
|          8|710301|
|          9|668097|
|         10|676128|
|         11|608818|
|         12|579565|
+-----------+------+



In [41]:
from pyspark.sql.functions import date_format

monthly_crime = (
    crime_clean_df
    .groupBy("Crime_Month")
    .agg(
        count("*").alias("Crime_Count")
    )
    .orderBy("Crime_Month")
)

monthly_crime.show()

+-----------+-----------+
|Crime_Month|Crime_Count|
+-----------+-----------+
|          1|     621870|
|          2|     547454|
|          3|     649788|
|          4|     641066|
|          5|     682819|
|          6|     681640|
|          7|     717118|
|          8|     710301|
|          9|     668097|
|         10|     676128|
|         11|     608818|
|         12|     579565|
+-----------+-----------+



In [42]:
hourly_crime = (
    crime_clean_df
    .groupBy("Crime_Hour")
    .count()
    .orderBy("Crime_Hour")
)

hourly_crime.show(24)

+----------+------+
|Crime_Hour| count|
+----------+------+
|         0|444272|
|         1|246808|
|         2|208146|
|         3|168308|
|         4|127941|
|         5|106840|
|         6|124613|
|         7|177870|
|         8|263432|
|         9|336467|
|        10|329969|
|        11|345202|
|        12|446899|
|        13|369553|
|        14|392468|
|        15|414378|
|        16|393014|
|        17|399856|
|        18|425556|
|        19|438502|
|        20|436963|
|        21|423861|
|        22|416908|
|        23|346838|
+----------+------+



In [43]:
hourly_crime.orderBy(
    col("count").desc()
).show(10)

+----------+------+
|Crime_Hour| count|
+----------+------+
|        12|446899|
|         0|444272|
|        19|438502|
|        20|436963|
|        18|425556|
|        21|423861|
|        22|416908|
|        15|414378|
|        17|399856|
|        16|393014|
+----------+------+
only showing top 10 rows


In [44]:
district_crime = (
    crime_clean_df
    .groupBy("District")
    .count()
    .orderBy(col("count").desc())
)

district_crime.show(20)

+--------+------+
|District| count|
+--------+------+
|       8|523193|
|      11|499808|
|       6|455120|
|       7|450420|
|      25|442880|
|       4|441996|
|       3|394841|
|      12|385665|
|       9|380367|
|       2|368384|
|      18|349314|
|      19|349312|
|       5|345021|
|      10|335394|
|      15|333750|
|       1|314598|
|      14|301290|
|      16|260482|
|      22|255159|
|      24|235064|
+--------+------+
only showing top 20 rows


In [45]:
arrest_analysis = (
    crime_clean_df
    .groupBy("Arrest")
    .count()
    .orderBy(col("count").desc())
)

arrest_analysis.show()

+------+-------+
|Arrest|  count|
+------+-------+
| false|5749900|
|  true|2034764|
+------+-------+



In [46]:
arrest_percentage = (
    arrest_analysis
    .withColumn(
        "Percentage",
        round((col("count") / total_records) * 100, 2)
    )
)

arrest_percentage.show()

+------+-------+----------+
|Arrest|  count|Percentage|
+------+-------+----------+
| false|5749900|     73.86|
|  true|2034764|     26.14|
+------+-------+----------+



In [47]:
arrest_by_type = (
    crime_clean_df
    .groupBy("Primary Type")
    .agg(
        count("*").alias("Total_Crimes"),
        sum(when(col("Arrest") == True, 1).otherwise(0)).alias("Arrests")
    )
    .withColumn(
        "Arrest_Rate",
        round((col("Arrests") / col("Total_Crimes")) * 100, 2)
    )
    .orderBy(col("Arrest_Rate").desc())
)

arrest_by_type.show(30, truncate=False)

+---------------------------------+------------+-------+-----------+
|Primary Type                     |Total_Crimes|Arrests|Arrest_Rate|
+---------------------------------+------------+-------+-----------+
|DOMESTIC VIOLENCE                |1           |1      |100.0      |
|PROSTITUTION                     |69840       |69567  |99.61      |
|NARCOTICS                        |747633      |743040 |99.39      |
|GAMBLING                         |14618       |14511  |99.27      |
|LIQUOR LAW VIOLATION             |14901       |14770  |99.12      |
|PUBLIC INDECENCY                 |194         |192    |98.97      |
|CONCEALED CARRY LICENSE VIOLATION|1080        |1040   |96.3       |
|INTERFERENCE WITH PUBLIC OFFICER |18392       |16905  |91.91      |
|OBSCENITY                        |818         |645    |78.85      |
|WEAPONS VIOLATION                |106418      |78880  |74.12      |
|CRIMINAL TRESPASS                |214316      |150299 |70.13      |
|OTHER NARCOTIC VIOLATION         

In [ ]:
#Geographic Analysis
geo_analysis = crime_clean_df.select(
    sum(
        when(
            col("Latitude").isNotNull() &
            col("Longitude").isNotNull(),
            1
        ).otherwise(0)
    ).alias("Valid_Geographic_Records"),
    
    sum(
        when(
            col("Latitude").isNull() |
            col("Longitude").isNull(),
            1
        ).otherwise(0)
    ).alias("Missing_Geographic_Records")
)

geo_analysis.show()

+------------------------+--------------------------+
|Valid_Geographic_Records|Missing_Geographic_Records|
+------------------------+--------------------------+
|                 7697816|                     86848|
+------------------------+--------------------------+



In [49]:
#Crime Concentration by Community Area
community_crime = (
    crime_clean_df
    .filter(col("Community Area").isNotNull())
    .groupBy("Community Area")
    .count()
    .orderBy(col("count").desc())
)

community_crime.show(20)

+--------------+------+
|Community Area| count|
+--------------+------+
|            25|448276|
|             8|252839|
|            43|236555|
|            23|223982|
|            28|217006|
|            24|210238|
|            29|209901|
|            67|205117|
|            71|203100|
|            49|190600|
|            68|187126|
|            69|178267|
|            32|177732|
|            66|174517|
|            44|158031|
|            22|148347|
|             6|144756|
|            61|144352|
|            26|135522|
|            27|134276|
+--------------+------+
only showing top 20 rows


In [50]:
#Crime Concentration by Beat
beat_crime = (
    crime_clean_df
    .filter(col("Beat").isNotNull())
    .groupBy("Beat")
    .count()
    .orderBy(col("count").desc())
)

beat_crime.show(20)

+----+-----+
|Beat|count|
+----+-----+
| 421|60663|
| 423|60105|
|1834|55488|
| 624|54753|
| 511|53401|
|1533|52543|
|1112|52409|
| 823|51475|
| 414|48991|
|1522|48487|
|2533|47464|
| 621|47277|
| 612|46541|
| 321|45231|
| 631|44564|
| 825|44359|
| 512|43516|
| 522|43308|
| 713|43139|
|1011|43121|
+----+-----+
only showing top 20 rows
